Predict speed based on TWD and TWS

In [2]:
import pandas as pd
import numpy as np
import sklearn
import seaborn as sns
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import pickle
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys

sys.path.append('/Users/hedge/dev/boats')

from boats.lib.common import *
from boats.lib.boat import Boat

In [3]:
df=Boat('Petsamo').log.df
df.reset_index(drop=True, inplace=True)
df['twa']=df['twa'].apply(lambda x: abs(x))
df=df[df['spd']!=0]
df.dropna(inplace=True)
df.tail()

/var/folders/0c/7w8qsw154p9gz8tn8b5fk1d80000gn/T/ipykernel_21875/116426813.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


,tws,spd,twd,twa,awa,heel,lat,lon,sails,hdg
1873,24.6,7.3,22,137,127.0,32,-55.73495,-66.42909,"[Mainsail, Mizzen, Nr.1]",246.0
1874,24.4,7.3,23,138,127.0,31,-55.76575,-66.52612,"[Mainsail, Mizzen, Nr.1]",246.0
1875,24.6,7.3,24,138,128.0,31,-55.81322,-66.67015,"[Mainsail, Mizzen, Nr.1]",246.0
1876,25.4,7.7,23,147,137.0,21,-55.82232,-66.69604,"[Mainsail, Mizzen, Nr.2]",236.0
1877,24.6,6.2,24,148,139.0,24,-55.83179,-66.71531,"[Mainsail, Mizzen, Nr.2]",236.0


In [4]:
len(df)

1374

In [5]:
estimators=[('ridgecv', RidgeCV()), ('lassocv', LassoCV()), ('lr', LinearRegression()), ('elasticcv', ElasticNetCV())]

In [6]:
X=df[['tws', 'twd', 'hdg']]
y=df['spd']

In [7]:
scores = []
for estimator in estimators:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.005, random_state=42) 
    for degree in range(10):
        pipe = make_pipeline(PolynomialFeatures(degree=degree), estimator[1])
        pipe.fit(X_train, y_train)
        scores.append( (estimator[0], degree, pipe.score(X_test, y_test)) )


In [8]:
df_scores = pd.DataFrame(scores, columns=['Estimator', 'Degree', 'Score'])
df_scores.set_index('Estimator', inplace=True)
df_scores.groupby('Estimator').max()

,Degree,Score
Estimator,,
elasticcv,9,-0.043749
lassocv,9,-0.045117
lr,9,0.359556
ridgecv,9,-0.053738


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.005, random_state=42) 
pipe=make_pipeline(PolynomialFeatures(degree=7), LinearRegression()).fit(X_train, y_train)
yhat=pipe.predict(X)
print(round(pipe.score(X, y),2))

0.73


In [10]:
df_lr=X.copy()
df_lr['Predicted']=[round(yhat[i],1) for i in range(len(yhat))]
df_lr['Actual']=y
df_lr

,tws,twd,hdg,Predicted,Actual
495,9.2,85,28.0,4.2,5.5
497,8.4,88,28.0,4.6,5.3
498,8.7,92,28.0,5.7,5.4
499,8.2,87,28.0,4.3,5.2
500,7.6,83,28.0,3.3,4.5
...,...,...,...,...,...
1873,24.6,22,246.0,6.7,7.3
1874,24.4,23,246.0,6.7,7.3
1875,24.6,24,246.0,6.8,7.3
1876,25.4,23,236.0,7.7,7.7


In [11]:
import keras
from keras.models import Sequential
from keras.layers import Dense

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
predictors=df[['tws', 'twd', 'hdg']]
target=df['spd']
model = Sequential()
model.add(Dense(200, activation='relu', input_shape=(len(predictors.columns),)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(predictors, target, validation_split=0.1, epochs=100)
yhat=model.predict(predictors)

In [ ]:
df_kr=predictors.copy()
df_kr['Predicted']=[round(yhat[i][0],1) for i in range(len(yhat))]
df_kr['Actual']=target
df_kr

In [ ]:
sns.scatterplot(df_lr.copy()[['Predicted', 'Actual', 'tws']].set_index('tws')).set_title('LR')

In [ ]:
sns.scatterplot(df_kr.copy()[['Predicted', 'Actual', 'tws']].set_index('tws')).set_title('Keras')

In [ ]:
sns.regplot(df_lr, x='tws', y='Actual')

In [ ]:
sns.regplot(df_lr, x='tws', y='Predicted')

In [ ]:
tws=15
twd=105
hdg=177
data=pd.DataFrame([[tws, twd, hdg]])

In [ ]:
round(pipe.predict(data)[0],1)

In [ ]:
round(model.predict(data)[0][0],1)